# Silverwing-ML: Full Training Pipeline on Colab GPU

Trains the 102M parameter Silverwing LLM on a T4 GPU using the **real repo**.

Pipeline:
1. **Generate expanded corpus** (step-by-step solutions, word problems, proofs, tutorials)
2. **Base Pretrain** (3000 steps) on expanded corpus → `best.pt`
3. **Continued pretrain** (3000 steps) → `cont/best.pt`
4. **SFT with chain-of-thought** (500 steps) → `sft-combined/best.pt`
5. Test generation
6. Download checkpoints

**Setup:** Runtime → Change runtime type → T4 GPU

In [ ]:
# Cell 1: Mount Drive
import os

from google.colab import drive

drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/silverwing'
os.makedirs(DRIVE, exist_ok=True)
print('Drive mounted:', DRIVE)

In [ ]:
# Cell 2: Install dependencies
!pip install -q torch --index-url https://download.pytorch.org/whl/cu121
!pip install -q pyyaml numpy
import torch

print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU - CHANGE RUNTIME!"}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB' if torch.cuda.is_available() else '')

In [ ]:
# Cell 3: Clone real repo (public)
!git clone https://github.com/oledesug-source/silverwing-ml.git /content/Silverwing-ML
os.chdir('/content/Silverwing-ML')
print(f'Working directory: {os.getcwd()}')
!git log --oneline -3

In [ ]:
# Cell 4: Generate expanded corpus + enhanced SFT data
!python scripts/expand_corpus.py --seed 42
!python scripts/build_combined_corpus.py

# Replace training corpus with combined version (trainer expects train.0.jsonl)
import shutil

shutil.copy('experiments/corpus/combined_train.jsonl', 'experiments/corpus/train.0.jsonl')
# Replace SFT with enhanced version (trainer reads sft-v1-combined.jsonl)
shutil.copy('experiments/sft/sft-v2-all.jsonl', 'experiments/sft/sft-v1-combined.jsonl')

n_corpus = sum(1 for _ in open('experiments/corpus/train.0.jsonl'))
n_sft = sum(1 for _ in open('experiments/sft/sft-v1-combined.jsonl'))
print(f'Corpus: {n_corpus} docs, SFT: {n_sft} records')

In [ ]:
# Cell 5: Patch configs for expanded corpus (skip hash verification)
import yaml

for cfg_path in ['configs/training.yaml', 'configs/training_cont.yaml']:
    with open(cfg_path) as f:
        cfg = yaml.safe_load(f)
    key = list(cfg.keys())[0]
    cfg[key]['verify_dataset'] = False
    cfg[key]['expected_dataset_hash'] = None
    cfg[key]['require_validation'] = False
    with open(cfg_path, 'w') as f:
        yaml.dump(cfg, f)
    print(f'Patched {cfg_path}')

## Stage 1: Base Pretrain (3000 steps on expanded corpus)
Output: `experiments/checkpoints/best.pt`

In [ ]:
# Cell 6: Base pretrain on expanded corpus
!python scripts/train.py \
    --config configs/training.yaml \
    --device cuda \
    --batch-size 4 \
    --max-steps 3000 \
    --no-clean-repo-check

## Stage 2: Continued pretrain (3000 steps)
Output: `experiments/checkpoints/cont/best.pt`

In [ ]:
# Cell 7: Continued pretrain
!python scripts/train.py \
    --config configs/training_cont.yaml \
    --device cuda \
    --batch-size 4 \
    --no-clean-repo-check

## Stage 3: SFT with Chain-of-Thought (500 steps)
Output: `experiments/checkpoints/sft-combined/best.pt`

In [ ]:
# Cell 8: SFT combined with chain-of-thought data
# Patch SFT config for more steps with larger dataset
import yaml

with open('configs/sft_combined.yaml') as f:
    cfg = yaml.safe_load(f)
cfg['sft']['max_steps'] = 500
cfg['sft']['eval_steps'] = 100
cfg['sft']['save_steps'] = 250
# dataset_path already points to sft-v1-combined.jsonl which we replaced
with open('configs/sft_combined.yaml', 'w') as f:
    yaml.dump(cfg, f)
print('SFT config patched: max_steps=500')

!python scripts/train_sft.py \
    --config configs/sft_combined.yaml \
    --device cuda \
    --no-clean-repo-check

## Stage 4: Test Generation

In [ ]:
# Cell 9: Test generation
import sys

import torch

sys.path.insert(0, '.')
from foundation.inference import Generator
from foundation.model.config import ModelConfig
from foundation.model.model import SilverwingDecoder
from foundation.tokenizer import TokenizerV2

tok = TokenizerV2.load('experiments/tokenizer')
print(f'Tokenizer vocab: {tok.vocab_size}')

ckpt_path = 'experiments/checkpoints/sft-combined/best.pt'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_cfg = ModelConfig.from_yaml('configs/model.yaml')
model = SilverwingDecoder(model_cfg)
state = torch.load(ckpt_path, map_location=device, weights_only=False)
model.load_state_dict(state.get('model_state', state))
model = model.to(device).eval()
print(f'Model: {sum(p.numel() for p in model.parameters())/1e6:.1f}M params')

gen = Generator(model, tok)
prompts = [
    'What is 2 + 2? ',
    'The answer to 3 * 3 is ',
    'Solve for x: 2x + 5 = 13. Step 1:',
    'Knowledge is ',
]
for p in prompts:
    result = gen.generate(p, max_new_tokens=80, temperature=0.7, top_k=50)
    print(f'Prompt: {p!r}')
    print(f'Output: {result.text!r}')
    print()

## Stage 5: Run Benchmark

In [ ]:
# Cell 10: Run math benchmark
!python scripts/run_benchmark.py \
    --model silverwing:experiments/checkpoints/sft-combined/best.pt \
    --benchmark math-benchmark-v1 \
    --output-dir experiments/eval

## Stage 6: Download Checkpoints

In [ ]:
# Cell 11: Download checkpoints individually (faster than zip)
from pathlib import Path

from google.colab import files

ckpt_dir = Path('experiments/checkpoints')
for f in ['sft-combined/best.pt', 'cont/best.pt', 'best.pt']:
    path = ckpt_dir / f
    if path.is_file():
        mb = path.stat().st_size / 1e6
        print(f'Downloading {f} ({mb:.0f} MB)...')
        files.download(str(path))
        print(f'  Done: {f}')

print('Save to: F:\\AI\\Silverwing-ML\\experiments\\checkpoints\\')